In [1]:
# ============================================================
# OFFICIAL TEST INFERENCE - Qwen2.5-0.5B-Instruct LoRA - GPU
# Kaggle: bật Accelerator = GPU
# Có resume: nếu bị ngắt thì chạy tiếp từ file output cũ
# ============================================================

!pip install -q transformers peft accelerate sentencepiece rouge_score pandas pyarrow evaluate

# ============================================================
# 0. ENV - ĐẶT TRƯỚC KHI IMPORT TORCH
# ============================================================

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import time
import torch
import pandas as pd
import evaluate

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# ============================================================
# 1. CONFIG
# ============================================================

TEST_PATH = "/kaggle/input/datasets/phuc23020636/dataset-test/test-00000-of-00001 (1).parquet"

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

ADAPTER_DIR = "/kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask"

OUTPUT_DIR = "/kaggle/working/official_test_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PRED_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_qwen2_5_0_5b_lora_predictions_gpu.csv"
)

OUTPUT_EVAL_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_qwen2_5_0_5b_lora_eval_gpu.csv"
)

OUTPUT_METRICS_PATH = os.path.join(
    OUTPUT_DIR,
    "official_test_qwen2_5_0_5b_lora_metrics_gpu.csv"
)

MAX_INPUT_TOKENS = 1500
MAX_OUTPUT_TOKENS = 200

# Giữ giống thí nghiệm cũ
NUM_BEAMS = 2

# Nếu muốn test nhanh trước thì đặt 5 hoặc 20.
# Chạy chính thức toàn bộ thì để None.
N_TEST_LIMIT = None


# ============================================================
# 2. DEVICE CHECK
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook chưa nhận GPU. Vào Kaggle Settings -> Accelerator -> GPU, "
        "Restart Session rồi chạy lại."
    )

DEVICE = torch.device("cuda")
DTYPE = torch.float16

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("Base model:", BASE_MODEL_NAME)
print("Adapter dir exists:", os.path.exists(ADAPTER_DIR))
print("Adapter model exists:", os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")))


# ============================================================
# 3. FIND ADAPTER IF PATH IS WRONG
# ============================================================

if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")):
    print("Không thấy adapter ở ADAPTER_DIR hiện tại:", ADAPTER_DIR)
    print("Đang tìm adapter Qwen 0.5B trong /kaggle/input ...")

    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" in files and "qwen2_5_0_5b" in root.lower():
            found.append(root)

    print("Các adapter tìm thấy:")
    for p in found:
        print(p)

    if len(found) > 0:
        ADAPTER_DIR = found[0]
        print("Tự động dùng adapter:", ADAPTER_DIR)
    else:
        raise FileNotFoundError("Không tìm thấy adapter_model.safetensors cho Qwen 0.5B.")

print("Adapter OK:", ADAPTER_DIR)


# ============================================================
# 4. READ TEST FILE
# ============================================================

def read_any(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file test: {path}")

    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    else:
        raise ValueError(f"Unsupported file type: {path}")


def clean_test_df(df, text_col="article", summary_col="summary"):
    df = df.copy()

    if text_col not in df.columns:
        raise ValueError(
            f"Không tìm thấy cột '{text_col}'. Columns hiện có: {df.columns.tolist()}"
        )

    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 50].reset_index(drop=True)

    if summary_col in df.columns:
        df[summary_col] = df[summary_col].astype(str).str.strip()
        df = df.rename(columns={text_col: "article", summary_col: "summary"})
        return df[["article", "summary"]].reset_index(drop=True)
    else:
        df = df.rename(columns={text_col: "article"})
        return df[["article"]].reset_index(drop=True)


raw_test_df = read_any(TEST_PATH)
official_test_df = clean_test_df(raw_test_df)

if N_TEST_LIMIT is not None:
    official_test_df = official_test_df.head(N_TEST_LIMIT).reset_index(drop=True)

print("Official test shape:", official_test_df.shape)
print("Columns:", official_test_df.columns.tolist())
display(official_test_df.head())


# ============================================================
# 5. QWEN PROMPT TEMPLATE
# ============================================================

def load_qwen_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME,
        trust_remote_code=True,
        use_fast=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"
    return tokenizer


def build_qwen_messages(article):
    return [
        {
            "role": "system",
            "content": (
                "Bạn là hệ thống tóm tắt văn bản tiếng Việt. "
                "Chỉ tóm tắt dựa trên văn bản được cung cấp, "
                "giữ lại các ý chính và không bịa thêm thông tin."
            )
        },
        {
            "role": "user",
            "content": (
                "Hãy tóm tắt văn bản sau thành một đoạn ngắn, "
                f"tối đa {MAX_OUTPUT_TOKENS} tokens.\n\n"
                f"{article}"
            )
        }
    ]


def build_qwen_prompt(article, tokenizer, add_generation_prompt=True):
    messages = build_qwen_messages(article)

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt
    )


# ============================================================
# 6. GENERATE WITH QWEN 0.5B LoRA ON GPU
# ============================================================

def generate_qwen_05_lora_gpu(
    test_df,
    base_model_name,
    adapter_dir,
    output_path
):
    print("=" * 80)
    print("Qwen2.5-0.5B LoRA OFFICIAL TEST INFERENCE - GPU")
    print("Base model:", base_model_name)
    print("Adapter:", adapter_dir)
    print("Test size:", len(test_df))
    print("Device:", DEVICE)
    print("NUM_BEAMS:", NUM_BEAMS)
    print("=" * 80)

    # Resume nếu file output đã tồn tại
    done_ids = set()
    results = []

    if os.path.exists(output_path):
        old_df = pd.read_csv(output_path)
        if "id" in old_df.columns:
            done_ids = set(old_df["id"].astype(int).tolist())
            results = old_df.to_dict("records")
            print(f"Resume mode: đã có {len(done_ids)} mẫu, sẽ chạy tiếp phần còn lại.")

    print("Loading tokenizer...")
    tokenizer = load_qwen_tokenizer()

    print("Loading base model on GPU FP16...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=DTYPE,
        device_map={"": 0},
        trust_remote_code=True
    )

    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(
        base_model,
        adapter_dir
    )

    model.eval()

    for i, row in test_df.iterrows():
        if int(i) in done_ids:
            continue

        article = str(row["article"])

        prompt = build_qwen_prompt(
            article,
            tokenizer=tokenizer,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt,
            max_length=MAX_INPUT_TOKENS,
            truncation=True,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        input_len = inputs["input_ids"].shape[1]

        torch.cuda.synchronize()
        start = time.time()

        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.float16):
                output_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=MAX_OUTPUT_TOKENS,
                    num_beams=NUM_BEAMS,
                    do_sample=False,
                    no_repeat_ngram_size=3,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

        torch.cuda.synchronize()
        elapsed = time.time() - start

        generated_ids = output_ids[0][input_len:]

        prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        item = {
            "id": int(i),
            "article": article,
            "prediction": prediction,
            "inference_time_sec": elapsed,
            "base_model": base_model_name,
            "adapter": adapter_dir,
            "mode": "qwen_0_5b_chat_lora_gpu",
            "num_beams": NUM_BEAMS
        }

        if "summary" in test_df.columns:
            item["reference"] = str(row["summary"])

        results.append(item)

        # Lưu sau mỗi mẫu để tránh mất tiến độ nếu Kaggle ngắt
        pd.DataFrame(results).sort_values("id").to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        completed = len(results)
        avg_time = sum(float(x["inference_time_sec"]) for x in results) / len(results)

        print(
            f"Done {completed}/{len(test_df)} | "
            f"current = {elapsed:.2f}s | avg = {avg_time:.2f}s/sample"
        )

        if completed % 20 == 0:
            torch.cuda.empty_cache()

    out_df = pd.DataFrame(results).sort_values("id").reset_index(drop=True)
    out_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("=" * 80)
    print("Saved prediction file:", output_path)
    print("Average time:", out_df["inference_time_sec"].mean())
    print("=" * 80)

    del model
    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    return out_df


pred_df = generate_qwen_05_lora_gpu(
    test_df=official_test_df,
    base_model_name=BASE_MODEL_NAME,
    adapter_dir=ADAPTER_DIR,
    output_path=OUTPUT_PRED_PATH
)

display(pred_df.head())


# ============================================================
# 7. COMPUTE ROUGE IF TEST HAS SUMMARY
# ============================================================

rouge = evaluate.load("rouge")

if "reference" in pred_df.columns:
    predictions = pred_df["prediction"].fillna("").astype(str).tolist()
    references = pred_df["reference"].fillna("").astype(str).tolist()

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False
    )

    metrics = {
        "model": "qwen2_5_0_5b_lora_official_test_gpu",
        "num_test_samples": len(pred_df),
        "rouge1": float(scores["rouge1"]),
        "rouge2": float(scores["rouge2"]),
        "rougeL": float(scores["rougeL"]),
        "rougeLsum": float(scores["rougeLsum"]),
        "avg_time_sec_per_sample": float(pred_df["inference_time_sec"].mean()),
        "num_beams": NUM_BEAMS,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0)
    }

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(
        OUTPUT_METRICS_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    eval_df = pd.DataFrame()
    eval_df["id"] = pred_df["id"]
    eval_df["prediction"] = pred_df["prediction"]
    eval_df["reference"] = pred_df["reference"]
    eval_df["source"] = pred_df["article"]

    eval_df.to_csv(
        OUTPUT_EVAL_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 80)
    print("OFFICIAL TEST METRICS")
    print(metrics)
    print("Saved eval file:", OUTPUT_EVAL_PATH)
    print("Saved metrics file:", OUTPUT_METRICS_PATH)

    display(metrics_df)

else:
    submit_df = pd.DataFrame()
    submit_df["id"] = pred_df["id"]
    submit_df["prediction"] = pred_df["prediction"]

    submit_path = os.path.join(
        OUTPUT_DIR,
        "official_test_qwen2_5_0_5b_lora_submit_gpu.csv"
    )

    submit_df.to_csv(
        submit_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Official test không có summary/reference.")
    print("Chỉ xuất file prediction để nộp:", submit_path)
    display(submit_df.head())


# ============================================================
# 8. SHOW OUTPUT FILES
# ============================================================

print("Output files:")
for f in os.listdir(OUTPUT_DIR):
    print(os.path.join(OUTPUT_DIR, f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


Qwen2.5-0.5B LoRA OFFICIAL TEST INFERENCE - GPU
Base model: Qwen/Qwen2.5-0.5B-Instruct
Adapter: /kaggle/input/datasets/phuc23020636/output-qwen05/checkpoints/qwen2_5_0_5b_lora_chatmask
Test size: 1344
Device: cuda
NUM_BEAMS: 2
Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model on GPU FP16...
Loading LoRA adapter...


...

Done 1/1344 | current = 10.98s | avg = 10.98s/sample
Done 2/1344 | current = 11.43s | avg = 11.21s/sample
Done 3/1344 | current = 9.91s | avg = 10.77s/sample
Done 4/1344 | current = 8.43s | avg = 10.19s/sample
Done 5/1344 | current = 9.35s | avg = 10.02s/sample
Done 6/1344 | current = 8.83s | avg = 9.82s/sample
Done 7/1344 | current = 7.07s | avg = 9.43s/sample
Done 8/1344 | current = 13.39s | avg = 9.92s/sample
Done 9/1344 | current = 7.65s | avg = 9.67s/sample
Done 10/1344 | current = 8.91s | avg = 9.60s/sample
Done 11/1344 | current = 9.47s | avg = 9.58s/sample
Done 12/1344 | current = 10.15s | avg = 9.63s/sample
Done 13/1344 | current = 8.39s | avg = 9.54s/sample
Done 14/1344 | current = 10.05s | avg = 9.57s/sample
Done 15/1344 | current = 7.82s | avg = 9.46s/sample
Done 16/1344 | current = 13.32s | avg = 9.70s/sample
Done 17/1344 | current = 9.69s | avg = 9.70s/sample
Done 18/1344 | current = 8.33s | avg = 9.62s/sample
Done 19/1344 | current = 12.24s | avg = 9.76s/sample
Done 20/1

,id,article,prediction,inference_time_sec,base_model,adapter,mode,num_beams,reference
0,0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip đã khai triển văn phòng new tại quận 7...,10.982319,Qwen/Qwen2.5-0.5B-Instruct,/kaggle/input/datasets/phuc23020636/output-qwe...,qwen_0_5b_chat_lora_gpu,2,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Dã quì bung rộn ràng trên các điểm đến của vườ...,11.427692,Qwen/Qwen2.5-0.5B-Instruct,/kaggle/input/datasets/phuc23020636/output-qwe...,qwen_0_5b_chat_lora_gpu,2,Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...",Tam Cốc là một điểm đến du lịch thu hút du khá...,9.909701,Qwen/Qwen2.5-0.5B-Instruct,/kaggle/input/datasets/phuc23020636/output-qwe...,qwen_0_5b_chat_lora_gpu,2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Nữ du khách Monica Rajesh đã trải nghiệm chuyế...,8.433427,Qwen/Qwen2.5-0.5B-Instruct,/kaggle/input/datasets/phuc23020636/output-qwe...,qwen_0_5b_chat_lora_gpu,2,Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...","Khu du khách CồnÉn nằm tại xã Taman Mỹ, huyện ...",9.354407,Qwen/Qwen2.5-0.5B-Instruct,/kaggle/input/datasets/phuc23020636/output-qwe...,qwen_0_5b_chat_lora_gpu,2,Cồn Én là một điểm đến du lịch nằm giữa sông T...


OFFICIAL TEST METRICS
{'model': 'qwen2_5_0_5b_lora_official_test_gpu', 'num_test_samples': 1344, 'rouge1': 0.6712562855275734, 'rouge2': 0.298168315991924, 'rougeL': 0.357390230656197, 'rougeLsum': 0.3597771615008124, 'avg_time_sec_per_sample': 9.617340689613705, 'num_beams': 2, 'max_input_tokens': 1500, 'max_output_tokens': 200, 'device': 'cuda', 'gpu_name': 'Tesla T4'}
Saved eval file: /kaggle/working/official_test_outputs/official_test_qwen2_5_0_5b_lora_eval_gpu.csv
Saved metrics file: /kaggle/working/official_test_outputs/official_test_qwen2_5_0_5b_lora_metrics_gpu.csv


,model,num_test_samples,rouge1,rouge2,rougeL,rougeLsum,avg_time_sec_per_sample,num_beams,max_input_tokens,max_output_tokens,device,gpu_name
0,qwen2_5_0_5b_lora_official_test_gpu,1344,0.671256,0.298168,0.35739,0.359777,9.617341,2,1500,200,cuda,Tesla T4


Output files:
/kaggle/working/official_test_outputs/official_test_qwen2_5_0_5b_lora_eval_gpu.csv
/kaggle/working/official_test_outputs/official_test_qwen2_5_0_5b_lora_metrics_gpu.csv
/kaggle/working/official_test_outputs/official_test_qwen2_5_0_5b_lora_predictions_gpu.csv
